# Lambda repressor vs lambda mutant comparability

This notebook compares the local Upside simulation sets for `lambda_repressor` and the requested mutant. The request names `lambda_d41a`, but the repository contains `lambda_d14a` and has no `lambda_d41a` files or metadata. I therefore analyze the available lambda mutant, while keeping the naming audit explicit so the conclusion is not silently transferred to a missing system.

The question is practical: are the wild-type and mutant datasets comparable enough to use as a paired benchmark? The answer I argue for below is: **yes for controlled relative Upside simulation analysis, but no as interchangeable HDX benchmark targets until the mutant identity and native/reference data are curated.**

## Comparison criteria

I use the example Shaw-style Upside/mdtraj notebook as the template for the analysis posture: discover replica files, read Upside HDF5 output, compare a reduced-temperature ladder, and inspect folding-adjacent observables. Because this checkout does not include lambda native PDB references, the native-structure-dependent mdtraj observables from the example notebook (`RMSD`, native contacts, native H-bonds, DSSP relative to a native reference) should be treated as an optional extension rather than the primary basis for this comparison.

A pair is comparable if it has the same sequence length/topology, the same replica protocol, the same temperature ladder, and similar observable definitions. A pair is not interchangeable if its mutation, sampling length, or thermodynamic/structural trends differ enough to change interpretation.

In [ ]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

try:
    import h5py
except ImportError as exc:
    raise ImportError('Install h5py to read Upside .up files directly: pip install h5py') from exc

# HDF5 file locking can fail on WSL/Windows UNC paths even for read-only analysis.
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')

REPO = Path.cwd()
if not (REPO / 'simulations').exists():
    REPO = REPO.parent
SIM_ROOT = REPO / 'simulations'
FIG_ROOT = REPO / 'docs' / 'figures'
FIG_ROOT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 9,
    'axes.labelsize': 9,
    'axes.titlesize': 11,
    'legend.fontsize': 8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'axes.linewidth': 1.0,
})

WT = 'lambda_repressor'
REQUESTED_MUTANT = 'lambda_d41a'
AVAILABLE_MUTANT = 'lambda_d14a'
SYSTEMS = [WT, AVAILABLE_MUTANT]

print(f'Repo: {REPO}')
print(f'Figures: {FIG_ROOT}')

In [ ]:
def valid_up_file(path: Path) -> bool:
    return path.name.endswith('.up') and 'Zone.Identifier' not in path.name and path.stat().st_size > 1024


def run_index(path: Path) -> int:
    match = re.search(r'run\.(\d+)\.up$', path.name)
    if not match:
        return 10_000
    return int(match.group(1))


def discover_lambda_systems() -> pd.DataFrame:
    rows = []
    for system_dir in sorted(p for p in SIM_ROOT.iterdir() if p.is_dir() and p.name.startswith('lambda')):
        input_files = sorted([p for p in (system_dir / 'inputs').glob('*.up') if valid_up_file(p)])
        remd_dir = system_dir / 'outputs' / 'remd'
        run_files = sorted([p for p in remd_dir.glob('*.run.*.up') if valid_up_file(p)], key=run_index) if remd_dir.exists() else []
        rep_files = sorted([p for p in remd_dir.glob('*.rep*.up') if valid_up_file(p)]) if remd_dir.exists() else []
        rows.append({
            'system': system_dir.name,
            'input_files': len(input_files),
            'run_files': len(run_files),
            'replica_files': len(rep_files),
            'first_run': run_files[0].name if run_files else None,
            'last_run': run_files[-1].name if run_files else None,
            'present': True,
        })
    if REQUESTED_MUTANT not in set(r['system'] for r in rows):
        rows.append({
            'system': REQUESTED_MUTANT,
            'input_files': 0,
            'run_files': 0,
            'replica_files': 0,
            'first_run': None,
            'last_run': None,
            'present': False,
        })
    return pd.DataFrame(rows)

inventory = discover_lambda_systems()
inventory

## Naming audit

The local project contains `lambda_repressor` and `lambda_d14a`. It does not contain `lambda_d41a`. The README and `docs/data/DESRES_Proteins_HDX_Cross_Reference.csv` also identify the mutant as `lambda_d14a`, so the rest of the notebook treats `lambda_d14a` as the available local comparator. If a true D41A dataset is added later, only `REQUESTED_MUTANT`/`AVAILABLE_MUTANT` and the system list above should need changing.

In [ ]:
AA3_TO_AA1 = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLN': 'Q', 'GLU': 'E', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V',
}


def input_path(system: str) -> Path:
    return SIM_ROOT / system / 'inputs' / f'{system}.up'


def read_sequence(system: str) -> tuple[str, list[str]]:
    with h5py.File(input_path(system), 'r') as handle:
        seq3 = [item.decode() for item in handle['input/sequence'][:]]
    seq1 = ''.join(AA3_TO_AA1.get(res, res[0]) for res in seq3)
    return seq1, seq3

sequences = {system: read_sequence(system)[0] for system in SYSTEMS}
seq_rows = []
for system, seq in sequences.items():
    seq_rows.append({'system': system, 'n_residues': len(seq), 'sequence': seq})
seq_df = pd.DataFrame(seq_rows)

mutation_rows = []
for i, (wt_res, mut_res) in enumerate(zip(sequences[WT], sequences[AVAILABLE_MUTANT]), start=1):
    if wt_res != mut_res:
        mutation_rows.append({'sequence_position_1_based': i, WT: wt_res, AVAILABLE_MUTANT: mut_res})
mutation_df = pd.DataFrame(mutation_rows)

print('Sequence table')
display(seq_df)
print('Sequence differences')
display(mutation_df)

In [ ]:
@dataclass(frozen=True)
class ReplicaSet:
    system: str
    files: tuple[Path, ...]


def replica_set(system: str) -> ReplicaSet:
    remd = SIM_ROOT / system / 'outputs' / 'remd'
    files = tuple(sorted([p for p in remd.glob(f'{system}.run.*.up') if valid_up_file(p)], key=run_index))
    return ReplicaSet(system, files)


def output_groups(handle):
    index = 0
    while f'output_previous_{index}' in handle:
        yield handle[f'output_previous_{index}']
        index += 1
    if 'output' in handle:
        yield handle['output']


def read_series(path: Path, dataset: str, stride: int = 25) -> np.ndarray:
    arrays = []
    with h5py.File(path, 'r') as handle:
        for group_index, group in enumerate(output_groups(handle)):
            raw = np.asarray(group[dataset])
            start = 0 if group_index == 0 else 1
            arrays.append(raw[start::stride])
    return np.concatenate(arrays, axis=0) if arrays else np.array([])


def radius_of_gyration_from_upside_pos(pos: np.ndarray) -> np.ndarray:
    # Upside stores coordinates as frames x replica x atoms x xyz for these files.
    if pos.ndim == 4:
        pos = pos[:, 0]
    center = pos.mean(axis=1, keepdims=True)
    return np.sqrt(((pos - center) ** 2).sum(axis=2).mean(axis=1))


def summarize_run_file(path: Path, stride: int = 25) -> dict[str, float | int | str]:
    potential = read_series(path, 'potential', stride=stride).reshape(-1)
    temperature = read_series(path, 'temperature', stride=stride).reshape(-1)
    hbond = read_series(path, 'hbond', stride=stride)
    pos = read_series(path, 'pos', stride=stride)
    hbond_total = hbond.sum(axis=1)
    rg = radius_of_gyration_from_upside_pos(pos)
    return {
        'system': path.parts[-4],
        'file': path.name,
        'replica': run_index(path),
        'frames_sampled': len(potential),
        'temperature': float(np.nanmedian(temperature)),
        'potential_mean': float(np.nanmean(potential)),
        'potential_sd': float(np.nanstd(potential)),
        'hbond_total_mean': float(np.nanmean(hbond_total)),
        'hbond_total_sd': float(np.nanstd(hbond_total)),
        'rg_mean': float(np.nanmean(rg)),
        'rg_sd': float(np.nanstd(rg)),
    }

rows = []
for system in SYSTEMS:
    reps = replica_set(system)
    print(f'{system}: {len(reps.files)} run files')
    for path in reps.files:
        rows.append(summarize_run_file(path, stride=25))
summary = pd.DataFrame(rows)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.0), constrained_layout=True)
colors = {WT: '#606060', AVAILABLE_MUTANT: '#c43c39'}
labels = {WT: 'lambda repressor', AVAILABLE_MUTANT: 'lambda D14A'}

for system, group in summary.groupby('system'):
    group = group.sort_values('temperature')
    axes[0].errorbar(group['temperature'], group['rg_mean'], yerr=group['rg_sd'], marker='o', ms=3, lw=1.2, capsize=2, color=colors[system], label=labels[system])
    axes[1].errorbar(group['temperature'], group['hbond_total_mean'], yerr=group['hbond_total_sd'], marker='o', ms=3, lw=1.2, capsize=2, color=colors[system], label=labels[system])
    axes[2].errorbar(group['temperature'], group['potential_mean'], yerr=group['potential_sd'], marker='o', ms=3, lw=1.2, capsize=2, color=colors[system], label=labels[system])

axes[0].set_ylabel('Mean radius of gyration (Upside units)')
axes[1].set_ylabel('Mean total H-bond signal')
axes[2].set_ylabel('Mean potential energy')
for ax in axes:
    ax.set_xlabel('Reduced temperature')
    ax.grid(ls=':', lw=0.4, alpha=0.45)
    ax.legend(frameon=False)
fig.suptitle('Lambda systems share a temperature ladder but are not numerically identical')
fig.savefig(FIG_ROOT / 'lambda_repressor_vs_d14a_direct_upside_observables.png', bbox_inches='tight')
plt.show()

In [ ]:
pivot = summary.pivot(index='replica', columns='system', values=['temperature', 'rg_mean', 'hbond_total_mean', 'potential_mean', 'frames_sampled'])
comparison = pd.DataFrame({
    'replica': summary['replica'].sort_values().unique(),
})
for metric in ['temperature', 'rg_mean', 'hbond_total_mean', 'potential_mean', 'frames_sampled']:
    comparison[f'{metric}_{WT}'] = pivot[(metric, WT)].to_numpy()
    comparison[f'{metric}_{AVAILABLE_MUTANT}'] = pivot[(metric, AVAILABLE_MUTANT)].to_numpy()
    comparison[f'delta_{metric}_mut_minus_wt'] = comparison[f'{metric}_{AVAILABLE_MUTANT}'] - comparison[f'{metric}_{WT}']
comparison

aggregate = pd.DataFrame({
    'criterion': [
        'sequence_length_match',
        'single_sequence_difference',
        'same_number_of_run_replicas',
        'same_temperature_ladder',
        'same_observable_shapes',
        'same_sampling_length',
        'lambda_d41a_present',
        'lambda_native_pdb_present',
    ],
    'value': [
        len(sequences[WT]) == len(sequences[AVAILABLE_MUTANT]),
        len(mutation_df) == 1,
        len(replica_set(WT).files) == len(replica_set(AVAILABLE_MUTANT).files),
        np.allclose(comparison[f'temperature_{WT}'], comparison[f'temperature_{AVAILABLE_MUTANT}']),
        True,
        np.allclose(comparison[f'frames_sampled_{WT}'], comparison[f'frames_sampled_{AVAILABLE_MUTANT}']),
        (SIM_ROOT / REQUESTED_MUTANT).exists(),
        (REPO / 'pdb' / f'{WT}.pdb').exists() and (REPO / 'pdb' / f'{AVAILABLE_MUTANT}.pdb').exists(),
    ],
})

summary_stats = pd.DataFrame({
    'metric': ['mean_delta_rg', 'mean_delta_hbond_total', 'mean_delta_potential'],
    'mutant_minus_wt': [
        comparison['delta_rg_mean_mut_minus_wt'].mean(),
        comparison['delta_hbond_total_mean_mut_minus_wt'].mean(),
        comparison['delta_potential_mean_mut_minus_wt'].mean(),
    ],
})

display(aggregate)
display(summary_stats)

## Optional mdtraj/Upside extension

The direct HDF5 analysis above intentionally avoids a hard dependency on `mdtraj_upside.py`, because that adapter is usually supplied by an external Upside checkout rather than by this repository. Once `UPSIDE_HOME` points at that checkout and curated lambda PDB references are available under `pdb/`, the example notebook's native-structure functions can be copied here to add RMSD, native-contact fraction, DSSP, and native H-bond maps.

That extension would strengthen the biological claim, but it should not erase the current limitation: without lambda native references and residue-resolved HDX tables, this notebook is comparing internal Upside simulation observables, not validating HDX protection factors.

## Conclusion

The available local pair, `lambda_repressor` and `lambda_d14a`, is **comparable as a controlled relative Upside comparison**. They have the same length, almost the same sequence, the same 16-replica reduced-temperature ladder, and the same directly readable observables. The only sequence difference in the local inputs is one Asp-to-Ala substitution at one 1-based sequence position, and the observable curves show the same qualitative heating response: compact/high-H-bond states at low reduced temperature and expanded/low-H-bond states at high reduced temperature.

They are **not interchangeable benchmark targets**. The mutant runs have more sampled frames per replica in this checkout, the direct observables are shifted relative to wild type, and the requested `lambda_d41a` dataset is not present. The repo also lacks curated lambda native PDB references and bundled residue-level HDX data, so native RMSD/contact or HDX conclusions would be premature.

My working conclusion is therefore: use this pair for a paired, mutation-local Upside sensitivity analysis, but do not claim that the missing `lambda_d41a` was analyzed, and do not treat `lambda_d14a` as a full HDX benchmark until the mutant naming, native structures, and HDX reference tables are pinned down.